In [1]:
import pandas as pd
from main import run_ga
from tqdm import trange
from sympy import parse_expr
import sympy as sp
import numpy as np
from pathlib import Path
from gen import generate_dataset

SEED = 56
NGUYEN_SAMPLE_SIZE = 2048
NGUYEN_DIR = Path("dataset/nguyen")
NGUYEN_EQS_PATH = NGUYEN_DIR / "eqs.csv"

# Ensure Nguyen benchmark data exists
if not NGUYEN_EQS_PATH.exists():
    print(f"Nguyen benchmark data not found at {NGUYEN_EQS_PATH}. Generating {NGUYEN_SAMPLE_SIZE} samples per equation...")
    generate_dataset(size=NGUYEN_SAMPLE_SIZE, seed=SEED, out_dir=NGUYEN_DIR)
else:
    print(f"Found existing Nguyen benchmark data at {NGUYEN_EQS_PATH}")

nguyen_specs = pd.read_csv(NGUYEN_EQS_PATH)
nguyen_specs.head()

df = pd.read_csv("dataset/FeynmanEquations.csv")
df.dropna(axis=0, how="all", inplace=True)
df.Formula = df.Formula.apply(lambda s: s.replace("arcsin", "asin").replace("arccos", "acos").replace("arctan", "atan"))
# dataset fixes/hacks present in original code
df.loc[18, "Filename"] = "I.15.10"
df.loc[82, "# variables"] = 3
df.loc[49, "Filename"] = "I.48.20"
df.loc[61, "Filename"] = "II.11.7"
df.head()

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
CUDA Version: 13.0
Found existing Nguyen benchmark data at dataset/nguyen/eqs.csv


,Filename,Number,Output,Formula,# variables,v1_name,v1_low,v1_high,v2_name,v2_low,...,v7_high,v8_name,v8_low,v8_high,v9_name,v9_low,v9_high,v10_name,v10_low,v10_high
0,I.6.2a,1.0,f,exp(-theta**2/2)/sqrt(2*pi),1.0,theta,1.0,3.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,I.6.2,2.0,f,exp(-(theta/sigma)**2/2)/(sqrt(2*pi)*sigma),2.0,sigma,1.0,3.0,theta,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,I.6.2b,3.0,f,exp(-((theta-theta1)/sigma)**2/2)/(sqrt(2*pi)*...,3.0,sigma,1.0,3.0,theta,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,I.8.14,4.0,d,sqrt((x2-x1)**2+(y2-y1)**2),4.0,x1,1.0,5.0,x2,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,I.9.18,5.0,F,G*m1*m2/((x2-x1)**2+(y2-y1)**2+(z2-z1)**2),9.0,m1,1.0,2.0,m2,1.0,...,2.0,z1,3.0,4.0,z2,1.0,2.0,NaN,NaN,NaN


In [ ]:
nguyen_unary_funcs = {
    'sin': (np.sin, sp.sin),
    'cos': (np.cos, sp.cos),
    'exp': (np.exp, sp.exp),
    'sqrt': (np.sqrt, sp.sqrt),
    'log': (np.log, sp.log),
    'abs': (np.abs, sp.Abs)
}

nguyen_results = []
n_nguyen = len(nguyen_specs)

for i in trange(n_nguyen, desc="Nguyen Benchmark"):
    row = nguyen_specs.iloc[i]
    data_path = Path(row['path'])
    if not data_path.is_absolute():
        data_path = Path.cwd() / data_path
    target_formula = row['equation'] if 'equation' in row.index else ''

    dataset = pd.read_csv(data_path)
    dataset = dataset.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    feature_cols = [col for col in dataset.columns if col != 'target']
    X = dataset[feature_cols].to_numpy()
    y = dataset['target'].to_numpy()

    best, best_score = run_ga(
        X,
        y,
        pop_size=1000,
        gens=100,
        max_init_depth=4,
        max_depth=10,
        crossover_rate=0.85,
        mutation_rate=0.2,
        tournament_k=5,
        elitism=12,
        var_names=feature_cols,
        unary_funcs=nguyen_unary_funcs,
        use_learned_crossover=True,
        selector_lr=0.0005,
        use_transformer=False,
        mlp_layers=15,
        mlp_hidden_dim=256,
        mlp_activation="gelu",
        mlp_dropout=0.1,
        stopping_metric="r2",
        stopping_threshold=1.0,
        complexity_coef=1e-2,
        inclusion_coef=0.4,
        prob_const=0.35,
        const_range=(-3, 3),
        point_mutation_prob=0.1,
        selector_update_freq=3,
        temperature=0.7,
        critic_weight=0.6,
        entropy_weight=0.02,
        verbose=True,
        print_freq=10,
        seed=SEED
    )

    nguyen_results.append({
        'equation_id': i + 1,
        'filename': data_path.name,
        'target_formula': target_formula,
        'found_formula': sp.expand(best.to_sympy()),
        'r2_score': best_score,
        'num_variables': len(feature_cols)
    })

    print(f"\n{'='*60}")
    print(f"Nguyen Equation {i+1}/{n_nguyen}: {data_path.name}")
    print(f"Target: {target_formula}")
    print(f"Found: {sp.expand(best.to_sympy())}")
    print(f"R² Score: {best_score:.6f}")
    print(f"{'='*60}\n")

nguyen_results_df = pd.DataFrame(nguyen_results)
print("\n" + "="*80)
print("NGUYEN BENCHMARK PERFORMANCE METRICS")
print("="*80)

if not nguyen_results_df.empty:
    print(f"\nTotal Equations Tested: {len(nguyen_results_df)}")
    print(f"Mean R² Score: {nguyen_results_df['r2_score'].mean():.6f}")
    print(f"Median R² Score: {nguyen_results_df['r2_score'].median():.6f}")
    print(f"Std Dev R² Score: {nguyen_results_df['r2_score'].std():.6f}")
    print(f"Min R² Score: {nguyen_results_df['r2_score'].min():.6f}")
    print(f"Max R² Score: {nguyen_results_df['r2_score'].max():.6f}")

    excellent = (nguyen_results_df['r2_score'] >= 0.99).sum()
    good = ((nguyen_results_df['r2_score'] >= 0.95) & (nguyen_results_df['r2_score'] < 0.99)).sum()
    moderate = ((nguyen_results_df['r2_score'] >= 0.90) & (nguyen_results_df['r2_score'] < 0.95)).sum()
    poor = (nguyen_results_df['r2_score'] < 0.90).sum()

    print(f"\nSuccess Rate Breakdown:")
    print(f"  Excellent (R² ≥ 0.99): {excellent}/{n_nguyen} ({100*excellent/n_nguyen:.1f}%)")
    print(f"  Good (0.95 ≤ R² < 0.99): {good}/{n_nguyen} ({100*good/n_nguyen:.1f}%)")
    print(f"  Moderate (0.90 ≤ R² < 0.95): {moderate}/{n_nguyen} ({100*moderate/n_nguyen:.1f}%)")
    print(f"  Poor (R² < 0.90): {poor}/{n_nguyen} ({100*poor/n_nguyen:.1f}%)")

    nguyen_latex_path = Path('papers/nguyen_results.tex')
    nguyen_table = nguyen_results_df.to_latex(
        index=False,
        float_format="%.6f",
        caption="Nguyen benchmark results",
        label="tab:nguyen_results"
    )
    nguyen_latex_path.write_text(nguyen_table)
    print(f"\nDetailed Nguyen results saved to '{nguyen_latex_path}'")
else:
    print("No Nguyen benchmark results were recorded.")
print("="*80)

Nguyen Benchmark:   0%|          | 0/12 [00:00<?, ?it/s]

Using learned subtree crossover with MLP (15 layers, 128 hidden, gelu activation, 0.1 dropout, A2C (with critic))
Stopping criterion: r2 >= 1.0


GA generations:   0%|          | 0/100 [00:00<?, ?gen/s]

Gen   1 best MSE+penalty: 156.352 R²: 0.939734 expr: (x * ((x * x) - sin(0.9139)))
Gen  10 best MSE+penalty: 0.846405 R²: 0.999998 expr: ((x * ((x * x) - sin(-1.806))) + (x * x))
Gen  10 best MSE+penalty: 0.846405 R²: 0.999998 expr: ((x * ((x * x) - sin(-1.806))) + (x * x))


Nguyen Benchmark:   8%|▊         | 1/12 [01:00<11:02, 60.19s/it]


Stopping criterion reached at generation 15
R²: 1 >= 1.0

Nguyen Equation 1/12: 1.csv
Target: x**3 + x**2 + x
Found: x**3 + x**2 + x
R² Score: 1.000000

Using learned subtree crossover with MLP (15 layers, 128 hidden, gelu activation, 0.1 dropout, A2C (with critic))
Stopping criterion: r2 >= 1.0


... (20 outputs left out)

GA generations:   0%|          | 0/100 [00:00<?, ?gen/s]

Gen   1 best MSE+penalty: 0.302937 R²: 0.930719 expr: x
Gen  10 best MSE+penalty: 0.298508 R²: 0.973676 expr: (1.168 * x)
Gen  10 best MSE+penalty: 0.298508 R²: 0.973676 expr: (1.168 * x)
Gen  20 best MSE+penalty: 0.298508 R²: 0.973676 expr: (1.168 * x)
Gen  20 best MSE+penalty: 0.298508 R²: 0.973676 expr: (1.168 * x)
Gen  30 best MSE+penalty: 0.294949 R²: 0.974734 expr: (x * 1.146)
Gen  30 best MSE+penalty: 0.294949 R²: 0.974734 expr: (x * 1.146)


Nguyen Benchmark:  50%|█████     | 6/12 [36:15<36:15, 362.51s/it]



KeyboardInterrupt: 

In [ ]:
n = len(df)
results = []
verbose = False  # Set to True to enable per-equation printing

for i in trange(n):
    vars_list = []
    for j in range(int(df["# variables"][i])):
        var_name = df[f"v{j+1}_name"][i]
        vars_list.append(sp.Symbol(var_name, real=True))
    
    expr = parse_expr(
        df.Formula[i], 
        local_dict={str(var): var for var in vars_list}, 
        evaluate=False
    )
    dataset = pd.read_csv(f"dataset/Feynman_with_units/{df['Filename'][i]}", header=None, sep=" ")
    dataset = dataset.sample(frac=1).reset_index(drop=True)
    dataset.dropna(axis=1, how="all", inplace=True)
    y = dataset.iloc[:, -1].to_numpy()
    X = dataset.iloc[:, :-1].to_numpy()
    unary_funcs = {
        'sin': (np.sin, sp.sin),
        'cos': (np.cos, sp.cos),
        'tan': (np.tan, sp.tan),
        'exp': (np.exp, sp.exp),
        'sqrt': (np.sqrt, sp.sqrt)
    }
    best, best_score = run_ga(
        X,
        y,
        pop_size=1500,
        gens=30,
        max_init_depth=4,
        max_depth=10,
        crossover_rate=0.85,
        mutation_rate=0.2,
        tournament_k=5,
        elitism=15,
        var_names=[str(var) for var in vars_list],  # Convert Symbols to strings
        unary_funcs=unary_funcs,
        use_learned_crossover=True,
        selector_lr=0.0005,
        use_transformer=False,
        mlp_layers=15,
        mlp_hidden_dim=256,
        mlp_activation="gelu",
        mlp_dropout=0.1,
        stopping_metric="r2",
        stopping_threshold=1.0,
        # NEW: Additional customization options
        complexity_coef=1e-4,
        inclusion_coef=0.5,
        prob_const=0.35,
        const_range=(-3, 3),
        point_mutation_prob=0.1,
        selector_update_freq=3,
        temperature=0.7,
        critic_weight=0.6,
        entropy_weight=0.02,
        verbose=True,
        print_freq=15,
        seed=SEED,
    )
    
    # Store results
    results.append({
        'equation_id': i + 1,
        'filename': df['Filename'][i],
        'target_formula': df.Formula[i],
        'found_formula': str(best),
        'r2_score': best_score,
        'num_variables': int(df["# variables"][i])
    })
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"Equation {i+1}/{n}: {df['Filename'][i]}")
        print(f"Target: {df.Formula[i]}")
        print(f"Found: {best}")
        print(f"R² Score: {best_score:.6f}")
        print(f"{'='*60}\n")

# Calculate and display aggregate metrics
results_df = pd.DataFrame(results)
print("\n" + "="*80)
print("FINAL DATASET PERFORMANCE METRICS")
print("="*80)
print(f"\nTotal Equations Tested: {len(results_df)}")
print(f"Mean R² Score: {results_df['r2_score'].mean():.6f}")
print(f"Median R² Score: {results_df['r2_score'].median():.6f}")
print(f"Std Dev R² Score: {results_df['r2_score'].std():.6f}")
print(f"Min R² Score: {results_df['r2_score'].min():.6f}")
print(f"Max R² Score: {results_df['r2_score'].max():.6f}")

# Success rate metrics
excellent = (results_df['r2_score'] >= 0.99).sum()
good = ((results_df['r2_score'] >= 0.95) & (results_df['r2_score'] < 0.99)).sum()
moderate = ((results_df['r2_score'] >= 0.90) & (results_df['r2_score'] < 0.95)).sum()
poor = (results_df['r2_score'] < 0.90).sum()

print(f"\nSuccess Rate Breakdown:")
print(f"  Excellent (R² ≥ 0.99): {excellent}/{n} ({100*excellent/n:.1f}%)")
print(f"  Good (0.95 ≤ R² < 0.99): {good}/{n} ({100*good/n:.1f}%)")
print(f"  Moderate (0.90 ≤ R² < 0.95): {moderate}/{n} ({100*moderate/n:.1f}%)")
print(f"  Poor (R² < 0.90): {poor}/{n} ({100*poor/n:.1f}%)")

# Save detailed results as LaTeX for paper-ready inclusion
feynman_latex_path = Path('papers/feynman_results.tex')
feynman_table = results_df.to_latex(
    index=False,
    float_format="%.6f",
    caption="Feynman benchmark results",
    label="tab:feynman_results"
)
feynman_latex_path.write_text(feynman_table)
print(f"\nDetailed results saved to '{feynman_latex_path}'")
print("="*80)

  0%|          | 0/100 [00:00<?, ?it/s]

Using learned subtree crossover with MLP (15 layers, 256 hidden, gelu activation, 0.1 dropout, A2C (with critic))
Stopping criterion: r2 >= 1.0


GA generations:   0%|          | 0/30 [00:00<?, ?gen/s]

Gen   1 best MSE+penalty: 0.00103031 R²: 0.973480 expr: tan((1.891 / (exp(theta) / (0.3924 / theta))))
Gen  15 best MSE+penalty: 0.000525923 R²: 0.974372 expr: exp((-1.36 * theta))


  1%|          | 1/100 [1:35:23<157:24:09, 5723.73s/it]

Gen  30 best MSE+penalty: 0.000514112 R²: 0.976775 expr: exp((-1.366 * theta))
Using learned subtree crossover with MLP (15 layers, 256 hidden, gelu activation, 0.1 dropout, A2C (with critic))
Stopping criterion: r2 >= 1.0


GA generations:   0%|          | 0/30 [00:00<?, ?gen/s]

Gen   1 best MSE+penalty: 0.503853 R²: -0.623559 expr: (0.1305 / sqrt((sqrt(2.11) / (-0.2952 + sigma))))
